In [ ]:
import pandas as pd

# 1. Load the data and perform a Dataset Audit
file_path = '03-lesson/Online Retail.xlsx'
df = pd.read_excel(file_path)

print("Initial Dataset Info:")
df.info()
print("\nMissing values before cleaning:")
print(df.isnull().sum())

print("\nDescriptive statistics for Quantity and UnitPrice (before cleaning):")
print(df[['Quantity', 'UnitPrice']].describe())

# 2. Clean the data
# Remove rows where CustomerID is missing
df_cleaned = df.dropna(subset=['CustomerID'])

# Filter out transactions where InvoiceNo starts with 'c' (cancellations)
df_cleaned = df_cleaned[~df_cleaned['InvoiceNo'].astype(str).str.startswith('C')]

# Filter out non-positive values for Quantity and UnitPrice
df_cleaned = df_cleaned[(df_cleaned['Quantity'] > 0) & (df_cleaned['UnitPrice'] > 0)]

print("\nMissing values after cleaning:")
print(df_cleaned.isnull().sum())

print("\nDescriptive statistics for Quantity and UnitPrice (after cleaning):")
print(df_cleaned[['Quantity', 'UnitPrice']].describe())

# 3. Create a TotalSum column
df_cleaned['TotalSum'] = df_cleaned['Quantity'] * df_cleaned['UnitPrice']

print("\nFirst 5 rows of the cleaned data with TotalSum:")
print(df_cleaned.head())
print("\nShape of the cleaned data:")
print(df_cleaned.shape)


In [ ]:
# 4. Calculate RFM Metrics
# Recency: Days since last purchase
# Frequency: Number of unique invoices per customer
# Monetary: Sum of TotalSum for each customer

# Convert InvoiceDate to datetime if not already
df_cleaned['InvoiceDate'] = pd.to_datetime(df_cleaned['InvoiceDate'])

# Calculate the last transaction date for each customer
last_purchase_date = df_cleaned.groupby("CustomerID")["InvoiceDate"].max()

# Find the most recent date in the entire dataset for reference
snapshot_date = df_cleaned['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate Recency in days
rfm_r = (snapshot_date - last_purchase_date).dt.days.reset_index()
rfm_r.columns = ["CustomerID", "Recency"]

# Calculate Frequency
rfm_f = df_cleaned.groupby("CustomerID")["InvoiceNo"].nunique().reset_index()
rfm_f.columns = ["CustomerID", "Frequency"]

# Calculate Monetary
rfm_m = df_cleaned.groupby("CustomerID")["TotalSum"].sum().reset_index()
rfm_m.columns = ["CustomerID", "Monetary"]

# Merge RFM components
rfm = rfm_r.merge(rfm_f, on="CustomerID")
rfm = rfm.merge(rfm_m, on="CustomerID")

print("\nRFM DataFrame Head:")
print(rfm.head())

print("\nRFM Descriptive Statistics:")
print(rfm.describe())
